In [ ]:
# global running var (keep true for program to work)
running = True

# dict to store income with description -> amount as the key -> value
income = {}
# dict to store income with category -> amount as the key -> value
expense = {}
budgetList = {}
# list to store each transaction, including its time stamp, amount, description, and if its an expense or income
transac = []
# balance being set to zero initially
balance = 0

jsonFilePath = "saveData.json"

# importning json to be able to save information when the program is reloaded
import json
# importing time to be able to delay execution of a line
import time
# importing datetime from the datetime module
from datetime import datetime
# importing os module to deal with files
import os

# function to save data
def saveData():
    # saving all variables in a dictionary  (some become nested dictionaries)
    data = {
        'income': income,
        'expense': expense,
        'budgetList': budgetList,
        'balance': balance,
        'transac': transac
    }

    # tries to create or edit the json file
    try:
        # with the file it opens budget data json file and writes over it if already existing at stores it as jsonFile var to refer back to later
        with open(jsonFilePath, 'w') as jsonFile:
            # converts the data we saved above into json format and dumps it into the jsonFile we created
            json.dump(data, jsonFile)
            print("Data saved successfully.")
    # if there is an error with the system, finding, creating or editing the file
    except OSError as e:
        # prints that there as an error and what it was
        print(f"File error while saving data: {e}")
    # if there was an error with the formating of the data that is being appended to the json file
    except TypeError as e:
        # prints that there was a data error and what it was
        print(f"Data format error: {e}")

# funciton to load data
def loadData():
    # declaring all vars
    global income, expense, balance, transac, budgetList
    # tries to open the json file in read mode
    try:
        with open(jsonFilePath, 'r') as jsonFile:
            # saves all data in the json file into the appropriate python variables
            data = json.load(jsonFile)
            income = data.get('income', {})
            expense = data.get('expense', {})
            transac = data.get('transac', [])
            budgetList = data.get('budgetList', {})
            balance = data.get('balance', 0)
            print("Data loaded successfully.")
    # if there is no such file yet it prints this
    except FileNotFoundError:
        print("No previous data found. Starting fresh.")
    except json.JSONDecodeError:
        print("Data file is corrupted. Data has not been loaded.")
    except:
        print("An error occured. Data has not been loaded")

# function to clear the saved data if the user wants to start fresh
def clearData():
    # declaring the variables to be loaded as global to take them from outside the function
    global income, expense, transac, balance, budgetList, expense, jsonFilePath
    # asks the user to confirm deleting the data and converts it to all lower case for handling
    confirm = str(input("Are you sure you want to delete all data? (yes/no)")).lower()
    # if the confirm string is yes
    if confirm == "yes":
        # clears dictionaries and lists and then sets balance to 0
        income.clear()
        expense.clear()
        transac.clear()
        budgetList.clear()
        balance = 0

        # tries to execute this so if there as an error it is cancelled
        try:
            # delete the JSON file if it exists
            if os.path.exists(jsonFilePath):
                os.remove(jsonFilePath)
                print("Data cleared and file deleted successfully.")
            else:
                print("Data cleared. No data file found to delete.")
        # if the os has an error with handling the file deletion
        except OSError as e:
            print(f"Could not delete data file: {e}")
    
    # if the confirmation is no then it cancels
    elif confirm == "no":
        print("Operation cancelled.")
    # if the input is anything else then it says that the input is invalid and cancels
    else:
        print("Invalid input. Operation cancelled.")
        
# parent class transaction
class transaction:
    # initializing the class with the description prompt amount prompt, and expense or income
    def __init__(self, descPrompt, amountPrompt, expORinc, label):
        # defining the description of self class as the user input (using the descPrompt as the title for the input)
        self.description = input(descPrompt)
        # repeats this loop until it is broken
        while True:
            # tries to take the input as a float
            try:
                # defining the amount of self class as the user input (using the amountPrompt as the title for the input)
                self.amount = float(input(amountPrompt))
                # raises the value error even if it is a float but not a positive num
                if self.amount <= 0:
                    raise ValueError
                # if it works then it breaks the loop and saves the ammount
                break
            # if not a float is entered then it prints that it needs to be a number and the loop repeats
            except ValueError:
                print("Please enter a valid positive number.")
                
        # defomomg the expire or income self class as the expire or income dictionaries
        self.expORinc = expORinc
        self.label = label

    # function to append it to a list with its own initiliaztions as its input
    def appendToList(self):
        # saves te current time in iso format (to save to the json)
        transacTime = datetime.now().isoformat()
        # appends the label, time, description, and amount to the transac list as a nested list
        transac.append([self.label, transacTime, self.description, self.amount])
        # checking if the description is not already in the dictionary
        if self.description not in self.expORinc:
            # adding the amount as the value with the description as the key inside the expire or income dictionary
            self.expORinc[self.description] = self.amount
        else:
            # adding the current amount to the already existing ammount with the description key in the expire or income dictionary
            self.expORinc[self.description] += self.amount

# child class addIncome (taken the output of the Transaction class as the input for this class)
class addIncome(transaction):
    # initializing the class
    def __init__(self):
        print("Adding income...")
        # initializing itself with the parents class initilizations, using these 3 attributes as the income for that init
        super().__init__("Enter income description:", "Enter amount ($):", income, "income")

        # calling the appendToList inside the parent class
        self.appendToList()
        # calling the append balance inside this class
        self.appendBalance()

    # function to appendBalance
    def appendBalance(self):
        global balance

        # adding the amount to the total balance
        balance += self.amount

        # prints that the income was added and the description and ammount
        print(f"Income Added: {self.description} - ${self.amount: .2f}")
        # prints the current balance
        print(f"Current Balance: ${balance: .2f}")

# child class addExpense (taken the output of the Transaction class as the input for this class)
class addExpense(transaction):
    # initializing the class
    def __init__(self):
        print("Adding expense...")
        # initializing itself with the parents class initilizations, using these 3 attributes as the income for that init
        super().__init__("Enter expense category:", "Enter amount ($):", expense, "expense")

        # if the description of the expense is in the budget list then...
        if self.description in budgetList:
            # call the append to list function (inside parent class)
            self.appendToList()
            # call the appendBalance function (inside this class)
            self.appendBalance()
        # otherwise it prints that the budget for this category is not yet defined and so it cannot add it
        else:
            print(f"{self.description} is not yet defined. Define the budget before you add expenses")

    # function to append balance
    def appendBalance(self):
        global balance
        # substracts the amount of the expense from the budget
        balance -= self.amount

        # storing the expense description key value as spent or 0 if it doesnt exist
        spent = expense.get(self.description, 0)
        # calculates the remaining amount of the budget
        remaining = budgetList[self.description] - spent
        # if the remaining budget is not a positive number
        if remaining < 0:
            # then it prints that the expenses have exceeded that categories budget
            print(f"Adding this expense has now exceeded your budget budget ")
            print(f"The {self.description} budget is ${abs(remaining):.2f} over budget")
        elif remaining == 0:
            print(f"The exact {self.description} budget limit has been reached of {budgetList[self.description]}$")
        else:
            remainPerc = remaining / budgetList[self.description]
            # prints the category and how much budget remains for that category
            print(f"Remaining {self.description} budget: ${remaining:.2f} ({remainPerc*100:.1f}%)")
            if remainPerc < 0.20:
                print("The budget limit is being approched. Less than 20% of the budget remains")
# function to view all transactions
def viewTrans():
    # tries to print all the entries
    try:
        # iterates through all transactions in the list
        for entry in transac:
            # converts the time from index 1 of entry back from iso format
            time = datetime.fromisoformat(entry[1])
            # formats it into year, month, day, hour, minute, second
            formatedTime = time.strftime("%Y-%m-%d %H:%M:%S")
            # prints the entry 
            print(f"{entry[0]} - {formatedTime} | {entry[2]} : {entry[3]: .2f}$")
    # if an error is present within the entries...
    except (ValueError, IndexError):
        print("Invalid transaction record found.")

# function to set the budget
def setBudget():
    print("setting budget...")
    # taking the user input for the category to edit or add
    budgetCat = str(input("Enter budget category to add or edit:"))

    # runs until the loop is broken
    while True:
        # tries to take the input as a float
        try:
            # taking the user input for the amount to set for that category budget
            amount = float(input(f"Enter amount to set for {budgetCat}:"))
            # raises the value error even if it is a float but is not a positive num
            if amount <= 0:
                raise ValueError
            # breaks once the conditions are met
            break
        # if it is not a float or was negative or 0 then it says invalid number
        except ValueError:
            print("Please enter a valid positive number.")
            
    # setting the budgetList's dictionary value to amount for the budget cat key
    budgetList[budgetCat] = amount

#class to view budget summary
class viewSum:
    #initializing the class
    def __init__(self):
        self.displaySummary()

    def displaySummary(self):
        global income
        global expense
        global balance
    
        print("=== Budget Summary ===\n")

        # Income
        total_income = sum(income.values())
        total_budget = sum(budgetList.values())
        print("Income: ")
        print(f"Total Income: ${total_income: .2f}\n")

        # Expenses by category
        print("Expenses by category: ")
        for category, amount in budgetList.items():
            spent = expense.get(category, 0)
            percent = (spent/amount) * 100 if amount > 0 else 0
            print(f"{category}: ${spent: .2f} / {amount: .2f} ({percent:g}% used)")

        # Total Expenses
        total_expenses = sum(expense.values())
        print(f"\nTotal Expenses: ${total_expenses: .2f}")

        # Balance
        balance = total_income - total_expenses
        print(f"Balance: ${balance: .2f}")

        print(f"Total Budget: ${total_budget: .2f}")

        overUnderAmnt = total_budget - total_expenses
        overUndrStr = "Under Budget" if overUnderAmnt >= 0 else "Over Budget"
        print(f"Over/Under Budget: {overUnderAmnt} ({overUndrStr})")
        
class monthlyReport:
    def __init__(self):
        self.generateReport()

    def generateReport(self):
        global income, expense, budget

        print("\n=== Monthly Financial Report ===")

        # Totals
        total_income = sum(income.values())
        total_expenses = sum(expense.values())
        total_budget = sum(budgetList.values())
        net_income = total_income - total_expenses
        savings_rate = (net_income / total_income * 100) if total_income > 0 else 0
        date = datetime.now().strftime("%B %d, %Y")
        print(f"Date: {date}")
        
        print(f"Total Income: ${total_income:.2f}")
        print(f"Total Expenses: ${total_expenses:.2f}")
        print(f"Net Income: ${net_income:.2f}")
        print(f"Savings Rate: {savings_rate:.1f}%\n")

        # Top spending categories
        print("Top Spending Categories:")
        if expense:
            print("Top 3 Expenses:")
            # Sort expenses from highest to lowest
            sorted_expenses = sorted(
                expense.items(),
                key=lambda x: x[1],
                reverse=True
            )
            # Display up to 3 highest expenses
            for index in range(min(3, len(sorted_expenses))):
                category, amount = sorted_expenses[index]
                percent = (amount / total_expenses * 100) if total_expenses > 0 else 0
                print(f"{index + 1}. {category}: ${amount:.2f} ({percent:.1f}%)")
        else:
            print("No expenses recorded.")

        # Budget status
        print("Budget Status:")
        if total_budget > 0:
            utilization = (total_expenses / total_budget * 100)
            print(f"Total budget utilization: {utilization:.1f}%")
        
            print("Categories over 80% of budget share:")
            found = False
            for cat, budget_amount in budgetList.items():  # loop through all budget categories
                spent = expense.get(cat, 0)  # get how much has been spent in that category, default 0
                percent = (spent / budget_amount * 100) if budget_amount > 0 else 0
                if percent >= 80:
                    print(f"- {cat}: {percent:.0f}% used (${spent:.2f} of ${budget_amount:.2f})")
                    found = True
            if not found:
                print("None")
        else:
            print("No budget set.")

# function to exit the program
def exit():
    # global running variable
    global running
    saveData()
    print("exiting application")
    # waits 3 seconds before setting running to false to give the user time to view that data was saved succusfully before the window closes
    time.sleep(3)
    # set running to false
    running = False

# function to view help
def viewHelp():
    print("Available commands:")
    # iterate through the menuOpt nested list and display the 0 index (command) and the 2 index (descriptions)
    for cmd in menuOpt:
        print(f"  {cmd[0]} : {cmd[2]}")

# using a list to store the menu options tupples, each containing the command, function/class to call and description
# using a tupple for these as they should not be changed ever
menuOpt = [
    ("add income", addIncome, "Add a your monthly income"),
    ("add expense", addExpense, "Add a monthly expense"),
    ("view transac", viewTrans, "View all transactions"),
    ("set budget", setBudget, "Set a monthly budget"),
    ("view summary", viewSum, "View your budget summary"),
    ("monthly report", monthlyReport, "View detailed monthly financial report"),
    ("clear data", clearData, "Clear all data including, incomes, expenses and budget"),
    ("save data", saveData, "Save all data including, incomes, expenses and budget"),
    ("exit", exit, "Exit the program"),
    ("help", viewHelp, "View available commands")
]

loadData()

# while the program is running
while running:
    print()
    # take input of the user and store as the command variable
    command = input("Enter command: ")

    # iterate through the menuOpt nested list
    for cmd in menuOpt:
        # when the command user input is equal to the 0 index of the current list...
        if cmd[0] == command:
            # then exeucte the current list 1 index
            print()
            # tries to execute the command
            try:
                cmd[1]()
            # this line will catch any errors which are not caught and addressed individually
            except Exception as e:
                # prints that an error occured and what the error was
                print(f"An unexpected error occurred while executing the command: {e}")
            # break the loop to stop looking for matches as one was already found
            break
    # if the for loop iterated through all values and no matches were found print that the command was not found and offer help
    else:
        print("Command not found (type help to view commands)")

No previous data found. Starting fresh.



Enter command:  set budget



setting budget...


Enter budget category to add or edit: fiid
Enter amount to set for fiid: 100


Enter command:  add expense


...

fiid 150 is not yet defined. Define the budget before you add expenses



Enter command:  add expense



Adding expense...


Enter expense category: fiid
Enter amount ($): 150


Adding this expense has now exceeded your budget budget 
The fiid budget is $50.00 over budget

